# Build Submission V3 Ensemble

This notebook reproduces `code/build_submission_v3_ensemble.py` and writes a Kaggle-ready submission CSV using only local data and locally trained model outputs.

In [ ]:
from pathlib import Path
import subprocess
import sys
import numpy as np
import pandas as pd

In [ ]:
# Paths and blend weights
PROJECT_ROOT = Path('/Users/ayushkumar/Desktop/final-four-analytics-challenge-26')
TRAIN_PATH = PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Training_Set2.0.csv'
TEST_PATH = PROJECT_ROOT / 'data' / 'raw' / 'NCAA_Seed_Test_Set2.0.csv'
V2_DIR = PROJECT_ROOT / 'submissions' / 'no_external_best'
V3_DIR = PROJECT_ROOT / 'submissions' / 'v3_no_external'
OUT_DIR = PROJECT_ROOT / 'submissions' / 'v3_ensemble_notebook'

# Blend weights (must sum to > 0)
WEIGHT_V2 = 0.60
WEIGHT_V3 = 0.40

# Tournament size used to construct per-season missing-seed sets
TOURNAMENT_SIZE = 68

In [ ]:
def ensure_inputs() -> None:
    v2_file = V2_DIR / 'submission_no_leak_v2.csv'
    v3_diag = V3_DIR / 'diagnostics_v3_no_external.csv'

    if not v2_file.exists():
        subprocess.run(
            [
                sys.executable,
                str(PROJECT_ROOT / 'code' / 'build_submission_v2.py'),
                '--include-bid-type',
                '--exclude-team',
                '--output-dir',
                str(V2_DIR),
            ],
            check=True,
        )

    if not v3_diag.exists():
        subprocess.run(
            [sys.executable, str(PROJECT_ROOT / 'code' / 'build_submission_v3.py')],
            check=True,
        )

In [ ]:
ensure_inputs()
print('Required local prediction inputs are available.')

In [ ]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
pred_v2 = pd.read_csv(V2_DIR / 'submission_no_leak_v2.csv')
diag_v3 = pd.read_csv(V3_DIR / 'diagnostics_v3_no_external.csv')

merged = (
    test[['RecordID', 'Season', 'Bid Type']]
    .merge(pred_v2, on='RecordID', how='left')
    .merge(diag_v3[['RecordID', 'SeedPredRaw']], on='RecordID', how='left')
)

if merged['Overall Seed'].isna().any() or merged['SeedPredRaw'].isna().any():
    raise ValueError('Missing inputs while merging v2 and v3 predictions')

print('Merged shape:', merged.shape)
merged.head()

In [ ]:
total = WEIGHT_V2 + WEIGHT_V3
if total <= 0:
    raise ValueError('WEIGHT_V2 + WEIGHT_V3 must be > 0')

w2 = WEIGHT_V2 / total
w3 = WEIGHT_V3 / total

score = w2 * merged['Overall Seed'].to_numpy(dtype=float) + w3 * merged['SeedPredRaw'].to_numpy(dtype=float)
selected = merged['Bid Type'].notna().to_numpy()
final = np.zeros(len(merged), dtype=float)

for season, idx in merged.groupby('Season').groups.items():
    season_idx = np.array(list(idx))
    season_selected = season_idx[selected[season_idx]]
    if len(season_selected) == 0:
        continue

    known = set(
        train[(train['Season'] == season) & train['Overall Seed'].notna()]['Overall Seed'].astype(int)
    )
    available = sorted([s for s in range(1, TOURNAMENT_SIZE + 1) if s not in known])

    order = np.argsort(score[season_selected])
    assigned = np.zeros(len(season_selected), dtype=float)

    if len(available) == len(season_selected):
        assigned[order] = np.array(available, dtype=float)
    else:
        # Defensive fallback for unexpected shape mismatch.
        avail = np.array(available, dtype=float)
        q = np.linspace(0, len(avail) - 1, len(season_selected)).round().astype(int)
        assigned[order] = np.sort(avail[q])

    final[season_selected] = assigned

print(f'Weights -> v2: {w2:.3f}, v3: {w3:.3f}')
print('Non-zero predictions:', int((final > 0).sum()))

In [ ]:
OUT_DIR.mkdir(parents=True, exist_ok=True)
out_path = OUT_DIR / 'submission_v3_ensemble_notebook.csv'

submission = pd.DataFrame({'RecordID': merged['RecordID'], 'Overall Seed': final})
submission.to_csv(out_path, index=False)

print('Wrote:', out_path)
print('Submission shape:', submission.shape)
print('Columns:', submission.columns.tolist())
submission.head(20)

In [ ]:
# Quick validation
assert submission.shape[0] == test.shape[0], 'Row count mismatch with test set'
assert list(submission.columns) == ['RecordID', 'Overall Seed'], 'Submission column mismatch'
assert submission['Overall Seed'].isna().sum() == 0, 'Submission has NaN seeds'

print('Validation passed.')